<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/4_nonseparability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 不可分函数（Non-Separable）与协方差矩阵自适应

如果分别优化每个变量就能得到全局最优解，这类函数称为可分函数（Separable Function）。前面使用的 Sphere 与轴对齐 Ellipsoid 都属于可分函数，它们又可以分别是良态（well-conditioned）或病态（ill-conditioned）的。

本章讨论不可分（Non-separable）函数，以及为什么完整协方差矩阵对于这类问题至关重要。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Separable-CMA 在不可分 Ellipsoid 上的行为

#### Ellipsoid 函数

下面改变 Hessian 条件数 $10^\texttt{EllipsoidCondition}$（例如让 $\texttt{EllipsoidCondition}=0,1,\dots,3$）并观察等高线。对于可分 Ellipsoid，椭圆等高线的主轴与坐标轴平行；各主轴尺度由代码中的权重 `w` 决定。

In [ ]:
EllipsoidCondition = 1
def ellipsoid(x):
    """各分量平方的加权和，最优解为 (0,...,0)。"""
    w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
    return np.sqrt(np.sum(w * x ** 2))

In [ ]:
dx, dy = 0.05, 0.05
y, x = np.mgrid[slice(-1, 1 + dy, dy), slice(-1, 1 + dx, dx)]
z = np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        z[i,j] = ellipsoid(np.array([x[i,j], y[i,j]]))

plt.figure()
CS = plt.contour(x, y, z)
plt.clabel(CS, inline=1, fontsize=10)
plt.grid()
plt.axis('equal')

#### 旋转 Ellipsoid（坐标旋转后的 Ellipsoid）

使用随机生成的正交矩阵 $R$，将目标函数定义为 $f(x)=f_\text{ellipsoid}(Rx)$，即可得到一个旋转后的不可分 Ellipsoid。

再次改变 `EllipsoidCondition` 并绘制等高线，会看到椭圆主轴不再与坐标轴平行。

In [ ]:
# Orthogonalization
def gram_schmidt(mat):
    """Return a matrix whose row is orthonormalized"""
    R = mat.copy()
    for i in range(R.shape[0]):
        for j in range(i):
            R[i] = R[i] - np.dot(R[i], R[j]) * R[j]
        R[i] /= np.linalg.norm(R[i])
    return R

In [ ]:
EllipsoidCondition = 1
N = 2

# Orthogonal Matrix (Rotation Matrix)
R = gram_schmidt(np.random.randn(N, N))

def rotated_ellipsoid(x):
    """各分量平方的加权和，最优解为 (0,...,0)。"""
    w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
    y = np.dot(R, x)
    return np.sqrt(np.sum(w * y ** 2))

In [ ]:
dx, dy = 0.05, 0.05
y, x = np.mgrid[slice(-1, 1 + dy, dy), slice(-1, 1 + dx, dx)]
z = np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        z[i,j] = rotated_ellipsoid(np.array([x[i,j], y[i,j]]))

plt.figure()
CS = plt.contour(x, y, z)
plt.clabel(CS, inline=1, fontsize=10)
plt.grid()
plt.axis('equal')

前一章学习的是 Separable-CMA，它只学习每个变量自身的方差。

In [ ]:
class SepCMAES(object):
    """带 CSA 的 SepCMA Evolution Strategy。"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        init_sigma : float
            初始步长
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值
        self.D = np.ones(self.N)

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

        # For CSA
        self.ps = np.zeros(self.N)
        self.cs = 4.0 / (self.N + 4.0)
        self.ds = 1.0 + self.cs
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N * self.N))
        self.mueff = 1.0 / np.sum(self.weights**2)
        # For CMA
        self.cmu = self.mueff / (4 * self.N + self.mueff)

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape) * np.sqrt(self.D)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_param(self):
        """更新参数。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        # 更新进化路径（累积均值移动）
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * np.dot(self.weights, (self.arx[idx] - self.mean)) / np.sqrt(self.D) / self.sigma
        # 更新协方差矩阵的对角元素
        self.D = (1 - self.cmu) * self.D + self.cmu * np.dot(self.weights, (self.arx[idx] - self.mean) ** 2) / self.sigma ** 2
        # 若进化路径长度大于随机函数下的期望，则增大步长
        self.sigma = self.sigma * np.exp(self.cs / self.ds * (np.linalg.norm(self.ps) / self.chiN - 1))
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))

在旋转 Ellipsoid 上进行实验。

In [ ]:
EllipsoidCondition = 6
N = 10
R = gram_schmidt(np.random.randn(N, N))
def rotated_ellipsoid(x):
    """各分量平方的加权和，最优解为 (0,...,0)。"""
    w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
    y = np.dot(R, x)
    return np.sqrt(np.sum(w * y ** 2))

es = SepCMAES(func=rotated_ellipsoid, init_mean=np.zeros(10), init_sigma=1, nsample=10)
maxiter = 1000
mean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
diagC = np.zeros((maxiter, es.N)) * np.nan
for i in range(maxiter):
    es.sample(); es.evaluate(); es.update_param()
    mean[i] = es.func(es.mean); sigmaN[i] = es.sigma * es.N; diagC[i] = es.D
plt.subplot(211); plt.semilogy(mean, '-b', label='f(mean)'); plt.semilogy(sigmaN, '--g', label='sigma*N'); plt.grid(); plt.legend(loc='best')
plt.subplot(212); plt.semilogy(diagC); plt.xlabel('no. of iterations'); plt.ylabel('D[i]'); plt.grid()

改变条件数并比较收敛曲线。

In [ ]:
EllCondArray = np.linspace(0, 6, num=13, endpoint=True)
for j in range(EllCondArray.shape[0]):
    EllipsoidCondition = EllCondArray[j]
    N = 10
    R = gram_schmidt(np.random.randn(N, N))
    def rotated_ellipsoid(x):
        """各分量平方的加权和，最优解为 (0,...,0)。"""
        w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
        y = np.dot(R, x)
        return np.sqrt(np.sum(w * y ** 2))
    es = SepCMAES(func=rotated_ellipsoid, init_mean=np.zeros(10), init_sigma=1, nsample=10)
    maxiter = 1000
    mean = np.zeros(maxiter) * np.nan
    for i in range(maxiter):
        es.sample(); es.evaluate(); es.update_param(); mean[i] = es.func(es.mean)
    plt.semilogy(mean, label='Cond=1e'+str(EllipsoidCondition))
plt.xlabel('no. of iterations'); plt.grid(); plt.legend(loc='best', fontsize='small', ncol=2)

观察结果：
* 条件数增大时，收敛速度显著下降。
* 表现与只使用 CSA-ES 时相近。

原因：
* Separable-CMA 只学习各变量方差，所以搜索分布始终是主轴与坐标轴平行的椭圆。
* 当目标函数等高线已经旋转、主轴不与坐标轴平行时，这种模型无法表示变量之间的相关性，因此难以生成真正沿狭长谷底方向的优秀候选解。

## 多元正态分布的性质

#### 生成方式、期望与协方差矩阵

服从 $d$ 维标准正态分布 $\mathcal{N}(0,I)$ 的随机向量 $z=(z_1,\dots,z_d)$，可以由 $d$ 个相互独立的 $z_i\sim\mathcal{N}(0,1)$ 组成，因此 $\mathbb{E}[z]=0$ 且 $\mathbb{E}[zz^\mathrm{T}]=I$。若 $x=m+\sigma z$，则 $x\sim\mathcal{N}(m,\sigma^2I)$，并有
$$\begin{aligned}
\mathbb{E}[x]&=m,\\
\mathbb{E}[(x-m)(x-m)^\mathrm{T}]&=\sigma^2I.
\end{aligned}$$
若 $\Sigma$ 为对角矩阵，则 $x=m+\sqrt{\Sigma}z$ 服从 $\mathcal{N}(m,\Sigma)$。对于一般情况，协方差矩阵必须是半正定对称矩阵，其唯一的半正定对称平方根满足 $\Sigma=\sqrt{\Sigma}\sqrt{\Sigma}$，同样可以用 $x=m+\sqrt{\Sigma}z$ 生成一般多元正态随机向量。

#### 基本性质
若相互独立的随机向量 $x_i\sim\mathcal{N}(m_i,\Sigma_i)$，则它们的加权和仍为正态分布：
$$
\sum_{i=1}^{k}w_ix_i\sim\mathcal{N}\left(\sum_{i=1}^{k}w_im_i,\sum_{i=1}^{k}w_i^2\Sigma_i\right).
$$
线性投影也保持正态性：若 $x\sim\mathcal{N}(m,\Sigma)$，则
$$Ax\sim\mathcal{N}(Am,A\Sigma A^\mathrm{T}),$$
特别地 $a^\mathrm{T}x\sim\mathcal{N}(a^\mathrm{T}m,a^\mathrm{T}\Sigma a)$。

对于联合正态变量，无相关与独立等价。一般分布中“独立必然无相关”，但“无相关并不必然独立”。因此若 $z\sim\mathcal{N}(0,I)$ 且 $B$ 为正交矩阵，则 $\tilde z=Bz$ 仍满足
$$\mathbb{E}[\tilde z\tilde z^\mathrm{T}]=BIB^\mathrm{T}=I,$$
所以旋转标准正态分布并不会改变它的分布。

#### 密度、等高线与特征值分解
假设 $\Sigma$ 正定，多元正态分布的密度为
$$
p(x;m,\Sigma)=((2\pi)^d\det(\Sigma))^{-1/2}\exp\left(-\frac12(x-m)^\mathrm{T}\Sigma^{-1}(x-m)\right).
$$
因此等密度面是以 $m$ 为中心的椭球
$$(x-m)^\mathrm{T}\Sigma^{-1}(x-m)=\gamma^2.$$
设 $\Sigma=B\Lambda B^\mathrm{T}$，其中 $\Lambda=\mathrm{diag}(\lambda_1,\dots,\lambda_d)$，$B=[b_1,\dots,b_d]$。椭球各主轴方向为 $\pm b_i$，长度与 $\gamma\sqrt{\lambda_i}$ 成比例，而且
$$\sqrt{\Sigma}=B\sqrt{\Lambda}B^\mathrm{T}.$$
于是采样可以写成
$$x=m+\sum_{i=1}^{d}\sqrt{\lambda_i}(b_i^\mathrm{T}z)b_i,$$
也就是沿协方差矩阵每个特征向量方向叠加方差为相应特征值的一维正态扰动。

#### 正态分布范数的集中现象
虽然正态密度在均值处最大，但在高维空间里，样本并不会大量堆在均值附近。对 $x-m=\sqrt{\Sigma}z$，有
$$(x-m)^\mathrm{T}\Sigma^{-1}(x-m)=\|z\|_2^2,$$
而 $\|z\|_2^2$ 服从自由度为 $d$ 的 $\chi^2$ 分布，其期望为 $d$、方差为 $2d$。因此 $\|z\|_2^2/d$ 的标准差为 $\sqrt{2/d}$，维数越高越集中在 1 附近。换言之，高维标准正态样本主要集中在半径约为 $\sqrt d$ 的超球壳层附近，而不是原点附近。

这个现象也可以从高维体积看出：半径 $(1-\epsilon)r$ 的球体积仅为半径 $r$ 球体积的 $(1-\epsilon)^d$ 倍。即使 $\epsilon$ 很小，当 $d$ 增大时该比例也会趋近 0，因此高维球的大部分体积都位于靠近表面的薄壳中。概率是密度与体积共同作用的结果，不能只根据中心处密度最大就认为样本集中在中心。

## 完整协方差更新：考虑变量间依赖

#### 基本思想
为了让搜索分布能够表示与坐标系无关的任意椭球形状，需要学习完整的 $N\times N$ 正定对称协方差矩阵，而不仅是其对角元素。

#### 更新方式
设 $\Sigma=\sigma^2C$，候选解按 $x_i\sim\mathcal{N}(m,\sigma^2C)$ 生成。排序后，完整协方差矩阵按
$$
C\leftarrow(1-c_\mu)C+c_\mu\sum_{i=1}^{\lambda}w_i\left(\frac{x_{i:\lambda}-m}{\sigma}\right)\left(\frac{x_{i:\lambda}-m}{\sigma}\right)^\mathrm{T}
$$
更新，其中 $c_\mu$ 为学习率，右侧 $m$、$\sigma$、$C$ 都是生成当前候选解时的旧参数。

其最大似然与自然梯度解释和 Separable-CMA 相同，只是现在也学习非对角相关项。对于随机目标函数，如果 $x_{i:\lambda}\sim\mathcal{N}(m,\sigma^2C)$，则
$$\mathbb{E}\left[\sum_iw_i\frac{x_{i:\lambda}-m}{\sigma}\left(\frac{x_{i:\lambda}-m}{\sigma}\right)^\mathrm{T}\right]=C,$$
因此该更新同样满足无偏性。

#### 程序

In [ ]:
class CMAES(object):
    """带 CSA 的 CMA Evolution Strategy。"""
    def __init__(self, func, init_mean, init_sigma, nsample):
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解目标函数值
        self.D = np.ones(self.N) # 协方差矩阵特征值
        self.B = np.eye(self.N)  # 协方差矩阵特征向量
        self.C = np.dot(self.B * self.D, self.B.T) # 协方差矩阵
        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1
        self.ps = np.zeros(self.N)
        self.mueff = 1.0 / np.sum(self.weights**2)
        self.cs = (2.0 + self.mueff) / (self.N + 3.0 + self.mueff)
        self.ds = 1.0 + self.cs + max(1.0, np.sqrt(self.mueff / self.N))
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N * self.N))
        self.cmu = self.mueff / (self.N ** 2 / 2 + self.N + self.mueff)
    def sample(self):
        """生成候选解。"""
        self.arz = np.random.normal(size=self.arx.shape)
        self.ary = np.dot(np.dot(self.arz, self.B) * np.sqrt(self.D), self.B.T)
        self.arx = self.mean + self.sigma * self.ary
    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]): self.arf[i] = self.func(self.arx[i])
    def update_param(self):
        """更新参数。"""
        idx = np.argsort(self.arf)
        # 更新进化路径
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * np.dot(self.weights, self.arz[idx])
        # 更新协方差矩阵
        self.C = (1 - self.cmu) * self.C + self.cmu * np.dot(self.ary[idx].T * self.weights, self.ary[idx])
        # 对协方差矩阵进行特征值分解
        self.D, self.B = np.linalg.eigh(self.C)
        # 若进化路径长度大于随机函数下期望，则增大步长
        self.sigma = self.sigma * np.exp(self.cs / self.ds * (np.linalg.norm(self.ps) / self.chiN - 1))
        self.mean += np.dot(self.weights, self.arx[idx] - self.mean)

#### 在 rotated Ellipsoid 上的行为

下面 `D` 表示协方差矩阵特征值，`diagC` 表示协方差矩阵对角元素（各变量方差）。

In [ ]:
EllipsoidCondition = 6
N = 10
R = gram_schmidt(np.random.randn(N, N))
es = CMAES(func=rotated_ellipsoid, init_mean=np.zeros(10), init_sigma=1, nsample=10)
maxiter = 1000
mean = np.zeros(maxiter) * np.nan; sigmaN = np.zeros(maxiter) * np.nan
D = np.zeros((maxiter, es.N)) * np.nan; diagC = np.zeros((maxiter, es.N)) * np.nan
for i in range(maxiter):
    es.sample(); es.evaluate(); es.update_param()
    mean[i]=es.func(es.mean); sigmaN[i]=es.sigma*es.N; D[i]=es.D; diagC[i]=np.diag(es.C)
plt.figure(figsize=(9,3)); plt.subplot(131); plt.semilogy(mean); plt.semilogy(sigmaN); plt.grid()
plt.subplot(132); plt.semilogy(D); plt.ylabel('D[i]'); plt.grid()
plt.subplot(133); plt.semilogy(diagC); plt.ylabel('diagC[i]'); plt.grid(); plt.tight_layout()

#### 在 separable Ellipsoid 上的行为

In [ ]:
EllipsoidCondition = 6
es = CMAES(func=ellipsoid, init_mean=np.zeros(10), init_sigma=1, nsample=10)
maxiter = 1000
mean=np.zeros(maxiter)*np.nan; sigmaN=np.zeros(maxiter)*np.nan
D=np.zeros((maxiter,es.N))*np.nan; diagC=np.zeros((maxiter,es.N))*np.nan
for i in range(maxiter):
    es.sample(); es.evaluate(); es.update_param()
    mean[i]=es.func(es.mean); sigmaN[i]=es.sigma*es.N; D[i]=es.D; diagC[i]=np.diag(es.C)
plt.figure(figsize=(9,3)); plt.subplot(131); plt.semilogy(mean); plt.semilogy(sigmaN); plt.grid()
plt.subplot(132); plt.semilogy(D); plt.ylabel('D[i]'); plt.grid()
plt.subplot(133); plt.semilogy(diagC); plt.ylabel('diagC[i]'); plt.grid(); plt.tight_layout()

* 在 Separable Ellipsoid 与 Rotated Ellipsoid 上，协方差矩阵特征值的演化基本一致。
* 但如果只看协方差矩阵的对角元素，两种情况会明显不同，因为旋转问题需要通过非对角相关项表达主轴方向。

#### 比较 separable Ellipsoid 与 rotated Ellipsoid

In [ ]:
EllCondArray = np.linspace(0, 6, num=13, endpoint=True)
for j in range(EllCondArray.shape[0]):
    EllipsoidCondition=EllCondArray[j]; N=10; R=gram_schmidt(np.random.randn(N,N))
    es=CMAES(func=rotated_ellipsoid,init_mean=np.zeros(10),init_sigma=1,nsample=10)
    mean=np.zeros(1000)*np.nan
    for i in range(1000): es.sample(); es.evaluate(); es.update_param(); mean[i]=es.func(es.mean)
    plt.semilogy(mean,label='Cond=1e'+str(EllipsoidCondition))
plt.grid(); plt.legend(loc='best',fontsize='small',ncol=2); plt.title('rotated ellipsoid')

In [ ]:
EllCondArray = np.linspace(0, 6, num=13, endpoint=True)
for j in range(EllCondArray.shape[0]):
    EllipsoidCondition=EllCondArray[j]; N=10
    es=CMAES(func=ellipsoid,init_mean=np.zeros(N),init_sigma=1,nsample=10)
    mean=np.zeros(1000)*np.nan
    for i in range(1000): es.sample(); es.evaluate(); es.update_param(); mean[i]=es.func(es.mean)
    plt.semilogy(mean,label='Cond=1e'+str(EllipsoidCondition))
plt.grid(); plt.legend(loc='best',fontsize='small',ncol=2); plt.title('separable ellipsoid')

与 Separable-CMA 不同，学习完整非对角协方差矩阵的 CMA 在 rotated Ellipsoid 和 separable Ellipsoid 上表现几乎一致。这体现了完整 CMA 的旋转不变性：算法能够学习问题真正的主轴方向，而不依赖外部坐标系。